# PyYAML — Working with YAML Files

In [ ]:
# Day 31 — PyYAML
# YAML (YAML Ain't Markup Language) is the MOST common config format
# in DevOps and MLOps — Kubernetes, Docker Compose, Ansible, GitHub Actions,
# Azure Pipelines, MLflow configs, DVC pipelines — ALL use YAML
#
# In Day 19 we learned JSON. YAML is similar but MORE readable:
#   JSON: {"name": "web-01", "cpu": 85}
#   YAML: name: web-01
#         cpu: 85
#
# Today we'll cover:
#   1. YAML syntax basics
#   2. Reading YAML files with PyYAML
#   3. Writing YAML files
#   4. Nested structures (dicts, lists, mixed)
#   5. Multi-document YAML files
#   6. Safe loading (security!)
#   7. Real-world: Kubernetes manifests, CI/CD configs
#
# Install: pip install pyyaml

### YAML Syntax Basics

In [ ]:
# YAML uses INDENTATION (like Python!) instead of braces/brackets
#
# Key-value pairs:       key: value
# Nested objects:         use indentation (2 spaces recommended)
# Lists:                  use - (dash + space)
# Comments:               # this is a comment
# Strings:                usually no quotes needed
# Booleans:               true / false (also yes / no)
# Numbers:                42, 3.14 (auto-detected)
# Null:                   null or ~

# Example YAML:
# app_name: payment-service
# version: v3.2.1
# replicas: 3
# debug: false
# ports:
#   - 8080
#   - 8443
# env:
#   DB_HOST: postgres-svc
#   CACHE_HOST: redis-svc

print('YAML is just structured data using indentation and dashes')

### Reading YAML Files

In [ ]:
# First, let's CREATE a YAML file to work with

yaml_content = """
app_name: payment-service
version: v3.2.1
environment: production
replicas: 3
debug: false

ports:
  - 8080
  - 8443

resources:
  cpu: 500m
  memory: 256Mi

env_vars:
  DB_HOST: postgres-svc
  DB_PORT: "5432"
  CACHE_HOST: redis-svc
  LOG_LEVEL: info

health_check:
  path: /health
  port: 8080
  interval_seconds: 30
  timeout_seconds: 5
"""

with open('config.yaml', 'w') as f:
    f.write(yaml_content)

print('config.yaml created!')

In [ ]:
# Reading YAML into a Python dictionary

import yaml

with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)    # safe_load is the RECOMMENDED function

print(type(config))     # <class 'dict'>  -> YAML becomes a Python dict!
print()

# Access values just like a dictionary
print(f"App: {config['app_name']}")
print(f"Version: {config['version']}")
print(f"Replicas: {config['replicas']}")
print(f"Debug: {config['debug']}")
print(f"Ports: {config['ports']}")
print(f"DB Host: {config['env_vars']['DB_HOST']}")
print(f"Health check path: {config['health_check']['path']}")

# YAML lists become Python lists
# YAML nested objects become nested dicts
# YAML booleans become Python booleans
# YAML numbers become Python ints/floats

In [ ]:
# IMPORTANT: safe_load vs load
#
# yaml.load()       -> UNSAFE! Can execute arbitrary Python code from YAML
# yaml.safe_load()  -> SAFE! Only loads data types (str, int, list, dict, etc.)
#
# ALWAYS use safe_load() unless you have a very specific reason not to
# This is a SECURITY best practice

# Reading a YAML string (not a file)
yaml_string = """
name: Krishna
age: 25
skills:
  - Python
  - DevOps
  - Kubernetes
"""

data = yaml.safe_load(yaml_string)    # works with strings too!
print(data)
# {'name': 'Krishna', 'age': 25, 'skills': ['Python', 'DevOps', 'Kubernetes']}

### Writing YAML Files

In [ ]:
# Convert a Python dict to YAML and write to file

import yaml

deployment = {
    'apiVersion': 'apps/v1',
    'kind': 'Deployment',
    'metadata': {
        'name': 'api-gateway',
        'namespace': 'production',
        'labels': {'app': 'api-gateway', 'version': 'v2.0'}
    },
    'spec': {
        'replicas': 3,
        'selector': {
            'matchLabels': {'app': 'api-gateway'}
        },
        'template': {
            'spec': {
                'containers': [{
                    'name': 'api-gateway',
                    'image': 'myregistry/api-gateway:v2.0',
                    'ports': [{'containerPort': 8080}]
                }]
            }
        }
    }
}

# Write to YAML file
with open('deployment.yaml', 'w') as f:
    yaml.dump(deployment, f, default_flow_style=False, sort_keys=False)

# Read it back to verify
with open('deployment.yaml', 'r') as f:
    print(f.read())

# default_flow_style=False -> block style (readable multi-line format)
# sort_keys=False -> preserve the order of keys

In [ ]:
# Converting YAML to JSON and vice versa
# This is very useful because some tools need JSON, others need YAML

import yaml
import json

# YAML to JSON
with open('config.yaml', 'r') as f:
    data = yaml.safe_load(f)

with open('config.json', 'w') as f:
    json.dump(data, f, indent=4)

print('YAML → JSON conversion done!')

# JSON to YAML
with open('config.json', 'r') as f:
    data = json.load(f)

with open('config_from_json.yaml', 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print('JSON → YAML conversion done!')

### Multi-Document YAML

In [ ]:
# Kubernetes uses multi-document YAML files
# Documents are separated by ---

multi_doc = """
apiVersion: v1
kind: Service
metadata:
  name: web-service
spec:
  ports:
    - port: 80
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: web-deployment
spec:
  replicas: 3
"""

with open('k8s_resources.yaml', 'w') as f:
    f.write(multi_doc)

# Load ALL documents
with open('k8s_resources.yaml', 'r') as f:
    docs = list(yaml.safe_load_all(f))    # safe_load_all for multiple docs

print(f'Found {len(docs)} documents')
for i, doc in enumerate(docs):
    if doc:    # skip None docs
        print(f'\nDocument {i}: {doc["kind"]} → {doc["metadata"]["name"]}')

In [ ]:
# Practice Questions:

# 1. Create a YAML config file for an ML training pipeline:
#    model: random_forest
#    hyperparams: {n_estimators: 100, max_depth: 5}
#    data: {train_path: data/train.csv, test_path: data/test.csv}
#    Read it back and print each section

# 2. Write a function yaml_to_json(yaml_path, json_path) that converts
#    any YAML file to JSON format

# 3. Create a multi-document YAML file with 3 Kubernetes resources
#    (Service, Deployment, ConfigMap). Read all 3 and print their kinds

# 4. Write a function update_config(yaml_path, key, value) that:
#    - Reads the YAML file
#    - Updates the specified key
#    - Writes it back
#    Test: update_config('config.yaml', 'replicas', 5)

### DevOps / AI-MLOps Practice

In [ ]:
# YAML is the LANGUAGE of DevOps configuration

# Exercise 1: Kubernetes Manifest Generator
# Write a function generate_k8s_deployment(app_name, image, replicas, port):
#   Returns a properly structured K8s Deployment dict
#   Saves it as {app_name}-deployment.yaml
# Test: generate_k8s_deployment('api-gateway', 'myrepo/api:v2', 3, 8080)

# Exercise 2: CI/CD Pipeline Config
# Create a YAML file that represents a GitHub Actions-style pipeline:
# name: ML Pipeline
# trigger: [push, pull_request]
# stages:
#   - name: test
#     steps: [checkout, install deps, run pytest]
#   - name: train
#     steps: [load data, train model, evaluate]
#   - name: deploy
#     steps: [build docker, push to registry, deploy to k8s]

# Exercise 3: ML Experiment Config Loader
# Create a YAML config for an ML experiment:
# experiment:
#   name: server_health_v2
#   model: xgboost
#   hyperparams:
#     n_estimators: 200
#     max_depth: 5
#     learning_rate: 0.1
#   data:
#     path: data/servers.csv
#     test_size: 0.2
#   mlflow:
#     tracking_uri: http://localhost:5000
#     experiment_name: server-health
# Write a function that reads this config and uses it to
# train a model with the specified hyperparameters